In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from openai import OpenAI
from dotenv import load_dotenv

In [3]:
load_dotenv() # to load variables from .env into environment
api_key = os.environ.get("OPENAI_API_KEY")
print("Key loaded:", api_key is not None)

client = OpenAI(api_key=api_key)

df = pd.read_csv("../data/processed/test_results_with_explanations.csv", index_col=0)
df.head()


Key loaded: True


,actual_class,fraud_probability,fraud_prediction,anomaly_score,anomaly_flag,Amount,Hour,Log_Amount,explanation_signals,severity,recommended_action,explanation_text
55186,0,0.018127,0,0.266560,0,62.56,13.0,4.151984,[],Low,No immediate action required,No significant suspicious signals detected.
98717,0,0.003399,0,0.269841,0,19.95,18.0,3.042139,[],Low,No immediate action required,No significant suspicious signals detected.
243200,0,0.015543,0,0.283048,0,49.99,18.0,3.931630,[],Low,No immediate action required,No significant suspicious signals detected.
17161,0,0.004977,0,0.263003,0,29.82,7.0,3.428164,[],Low,No immediate action required,No significant suspicious signals detected.
355,0,0.022539,0,0.218786,0,5.46,0.0,1.865629,['transaction occurred during late-night hours'],Low,No immediate action required,Transaction occurred during late-night hours.


#### Build the GPT function

In [4]:
def generate_gpt_summary(row):
    prompt = f"""
You are a fraud analyst assistant.

Write a conise investigation summary (2 to 3 sentence max).

Guidelines:
- Be brief and direct
- Do NOT repeat numbers excessively
- Focus on key risk factors only
- Use professional, analyst-style language
- Avoid phrases like "this transaction shows"
- Avoid bullet points or numbering
- Vary sentence structure slightly across summaries
- Combine multiple risk factors into a single concise explanation
- Mention time only if relevant

Transactions details:
- Fraud probability: {row['fraud_probability']:.2f}
- Anomaly flag: {row['anomaly_flag']}
- Amount: {row['Amount']:.2f}
- Hour: {row['Hour']}
- Severity {row['severity']}
- Explanation {row['explanation_text']}
- Recommended action {row['recommended_action']}

Return only the summary text.
"""
    response = client.responses.create(
        model="gpt-5.4-nano",
        input=prompt,
        max_output_tokens=120
    )

    return response.output_text

#### Test on some sample records

In [5]:
pd.set_option('display.max_colwidth', None)
sample_df = df[df["severity"].isin(["High", "Critical"])].head(5).copy()
sample_df["gpt_summary"] = sample_df.apply(generate_gpt_summary, axis=1)
sample_df[[
    "actual_class",
    "fraud_probability",
    "anomaly_flag",
    "Amount",
    "Hour",
    "severity",
    "recommended_action",
    "gpt_summary"
]]

,actual_class,fraud_probability,anomaly_flag,Amount,Hour,severity,recommended_action,gpt_summary
151011,1,1.000000,1,1.00,2.0,Critical,Escalate immediately for fraud analyst review,"Very high fraud probability with a critical anomaly flag indicates unusual transaction behavior, supported by multiple strong risk signals and low anomaly score inconsistent with expected patterns. The activity occurred during late-night hours, warranting immediate escalation to fraud analyst review."
154429,0,0.709571,0,3026.52,4.0,High,Send for priority analyst review,"The transaction presents elevated fraud risk with a high fraud probability and high severity, further compounded by a relatively large value processed during late-night hours. Despite no anomaly flag, these combined indicators warrant priority analyst review to validate legitimacy and prevent potential loss."
155296,0,0.904657,1,102.24,5.0,Critical,Escalate immediately for fraud analyst review,"High fraud likelihood is indicated by a critical severity anomaly, with multiple strong risk signals and an anomalous pattern inconsistent with typical behavior. The transaction also occurred during late-night hours, warranting immediate escalation to a fraud analyst for further review."
10250,0,0.938471,0,26.99,4.0,High,Send for priority analyst review,"High fraud likelihood (0.94) with high severity, occurring during late-night hours, aligns with anomalous behavioral risk despite no anomaly flag being raised. Recommended for priority analyst review to confirm legitimacy and prevent potential account misuse."
219182,0,0.861950,1,4692.65,15.0,High,Send for priority analyst review,"High fraud probability (0.86) with an anomalous behavior flag and high severity indicates multiple strong risk signals, including unusual activity patterns reflected by the low anomaly score. Due to the relatively high transaction amount, this case warrants priority analyst review and potential escalation."


In [8]:
normal_cases = df[
    (df["actual_class"] == 0) &
    (df["severity"] == "Low")
].sample(3, random_state=912)

hidden_fraud = df[
    (df["actual_class"] == 1) &
    (df["anomaly_flag"] == 0) &
    (df["fraud_probability"] > 0.8)
].sample(3, random_state=912)

fraud_cases = df[
    (df["actual_class"] == 1) &
    (df["Amount"] > 0)
].sample(3, random_state=912)

high_risk_cases = df[
    (df["actual_class"] == 1) &
    (df["anomaly_flag"] == 1) &
    df["severity"].isin(["High", "Critical"])
].sample(3, random_state=912)

false_positive = df[
    (df["actual_class"] == 0) &
    (df["severity"].isin(["High", "Critical"]))
].sample(3, random_state=912)

presentation_cases = pd.concat([normal_cases, hidden_fraud, fraud_cases, high_risk_cases, false_positive]).drop_duplicates()
presentation_cases["gpt_summary"] = presentation_cases.apply(generate_gpt_summary, axis=1)
presentation_cases.to_csv("../data/processed/test_results_with_summaries.csv", index=True)